# Artefacts in ODYSSEY (H2)

H2 (#19). The methods come from the literature collected in `docs/method.md`, *Cleaning of ODYSSEY (H2)*;
the review of #74 chose the EURADCLIM filters that run on the composite alone (Gabella, `histo_cut` on past
composites), a threshold for impossible values, and Wagner-style frequency maps for the catalogue.

1. **Gabella on rain** (review of #74, point 5): how much convective rain the filter removes on summer
   convective days, at 2 km (ODYSSEY grid) and at 1 km (CIRRUS), and where the removed rain lies.
2. **Gabella on the January features**: how much it removes on the January 2025 days whose `lk` deficit
   came from stationary features centred on radars (H1, `docs/method.md`).
3. **Pixels above 55 and 70 dBZ** on ODYSSEY and CIRRUS (review of #74, impossible values).
4. **Catalogue:** frequency maps of reflectivity classes and `histo_cut` on annual sums, per year.

In [ ]:
from opera_ds.paths import DATA_DIR, RESULTS_DIR

import csv
import glob
from collections import defaultdict

import numpy as np
from scipy import ndimage
from wradlib.classify import filter_gabella

from opera_ds.classify import obs_classes
from opera_ds.odim_io import index_files, read_odim_composite
from opera_ds.validate import read_degraded

## Settings

- **Data:** CIRRUS over the Belgian crop at :00, :15, :30 and :45. At 2 km it is degraded by `zmean`, the H1
  stand-in for ODYSSEY; at 1 km it is the real CIRRUS.
- **Filter:** `wradlib.classify.filter_gabella` with its defaults (5 × 5 window, `tr1` = 6 dB, `n_p` = 6,
  `tr2` = 1.3), `cartesian=True`, as in EURADCLIM (Overeem et al. 2023, section 3.1). Field in dBZ,
  `undetect` and values below 0 dBZ set to 0 dBZ (`thrsnorain` = 0), NaN outside coverage.
- **Classes:** Steiner on the 2 km grid, as in H1; at 1 km each 2 km class is copied to its 2 × 2 block.
- **Rain:** Marshall–Palmer, dBZ capped at 55 (Imhoff et al. 2020), on echoes ≥ 10 dBZ (the wet threshold of
  `run_validation`).
- **Where the removed rain lies:** each removed convective pixel is put in one of three groups.
  - *small echo:* its echo region (echoes above 0 dBZ, 8-connected) has at most 11 pixels, the size below
    which the area/perimeter test removes a region (Gabella and Notarpietro 2002, section 2);
  - *edge:* otherwise, when one of its 8 neighbours has no echo;
  - *interior:* the rest.

In [ ]:
WET_DBZ = 10.0
SMALL_PX = 11
EIGHT = np.ones((3, 3), bool)

files = index_files(str(DATA_DIR / "cirrus_bel" / "**" / "*@DBZH.h5"), r"@(\d{8}T\d{4})@", "%Y%m%dT%H%M")


def rain(dbz):
    return (10 ** (np.minimum(dbz, 55.0) / 10) / 200.0) ** (1 / 1.6)


def analyse(path, km):
    """Sums of rain and pixels, flagged or not, per class, and of removed convective rain per position."""
    if km == 2:
        f = read_degraded(path, factor=2).to_field(0.0)
        cls = obs_classes(f, floor=0.0, factor=1, pixel_km=2.0)
    else:
        f = read_odim_composite(path, "DBZH").to_field(0.0)
        cls = obs_classes(f, floor=0.0, factor=2, pixel_km=1.0)
    flag = filter_gabella(f.copy(), wsize=5, cartesian=True)
    echo = np.isfinite(f) & (f > 0)
    wet = np.isfinite(f) & (f >= WET_DBZ)
    r = np.where(wet, rain(np.nan_to_num(f)), 0.0)
    out = {}
    for name, m in (("all", wet), ("convective", wet & cls["convective"]), ("stratiform", wet & cls["stratiform"])):
        out[name] = np.array([r[m].sum(), r[m & flag].sum(), m.sum(), (m & flag).sum()], float)
    lab, n = ndimage.label(echo, structure=EIGHT)
    size = np.bincount(lab.ravel())
    small = echo & (size[lab] <= SMALL_PX)
    edge = echo & ~small & (ndimage.minimum_filter(echo.astype(np.uint8), footprint=EIGHT, mode="constant", cval=1) == 0)
    rem = wet & cls["convective"] & flag
    out["position"] = np.array([r[rem & small].sum(), r[rem & edge].sum(), r[rem & ~small & ~edge].sum()])
    return out


def run(days):
    tot = {km: defaultdict(lambda: defaultdict(float)) for km in (2, 1)}
    todo = [(f"{t:%Y-%m-%d}", str(p)) for t, p in sorted(files.items()) if t.minute % 15 == 0 and f"{t:%Y-%m-%d}" in days]
    print(len(todo), "frames")
    for km in (2, 1):
        for d, path in todo:
            for k, v in analyse(path, km).items():
                for key in (d, "all days"):
                    tot[km][key][k] = tot[km][key][k] + v
    return tot


def report(tot, days):
    for km in (2, 1):
        print(f"--- {km} km")
        print(f"{'day':12s} {'class':11s} {'rain removed':>13s} {'pixels removed':>15s}")
        for d in list(days) + ["all days"]:
            for k in ("all", "convective", "stratiform"):
                s, sr, n, nr = tot[km][d][k]
                print(f"{d:12s} {k:11s} {sr / s:12.1%} {nr / n:14.1%}")
        p = tot[km]["all days"]["position"]
        print("removed convective rain: small echo {:.0%}, edge {:.0%}, interior {:.0%}".format(*(p / p.sum())))

## 1. Summer convective days

The 5 days of June–July 2025 with most Steiner convective pixels in the H1 run (rows at `alpha` = 0, which
are the same for every method).

In [ ]:
days = defaultdict(int)
for mon in ("20250601-20250701", "20250701-20250801"):
    with open(RESULTS_DIR / "h1_degraded_cirrus" / mon / "per_time.csv", encoding="utf-8") as fh:
        for r in csv.DictReader(fh):
            if r["method"] == "nearest|convective" and float(r["alpha"]) == 0.0:
                days[r["time"][:10]] += int(r["n_wet"])
summer = sorted(days, key=lambda d: -days[d])[:5]
print("days (convective pixels, H1, alpha = 0):", [(d, days[d]) for d in summer])
tot_summer = run(summer)
report(tot_summer, summer)

## 2. January days with stationary features

The 5 days that carry 70% of the January `lk` deficit on convective pixels, with stationary features centred
on or near radars (`notebooks/03-mr-cirrus-analysis.ipynb`, section 4). Here the "convective" pixels are
mostly those features, so what the filter removes is a gain, not a loss.

In [ ]:
with open(RESULTS_DIR / "cirrus_analysis" / "selected_lk_deficit_times.csv", encoding="utf-8") as fh:
    january = [r["time"][:10] for r in csv.DictReader(fh)]
print("days:", january)
tot_january = run(january)
report(tot_january, january)

## 3. Pixels above 55 and 70 dBZ

Echo pixels (covered, not `undetect`) of the Belgian crops on the project PC: ODYSSEY 2023 and CIRRUS 2025.
About 40 min in one process for CIRRUS; the counts of the run of 9/11/2026 are in `docs/method.md`,
*Impossible values: counts on ODYSSEY and CIRRUS*. Set `RUN = True` to recount.

In [ ]:
RUN = False


def count(path):
    fr = read_odim_composite(path, "DBZH")
    v = fr.values[fr.coverage & ~fr.undetect]
    return np.array([v.size, ((v >= 55) & (v <= 70)).sum(), (v > 70).sum(), (v >= 100).sum(), int((v > 70).any()), 1])


if RUN:
    for name, pat in (("ODYSSEY 2023", "odyssey_bel/2023/**/*.h5"), ("CIRRUS 2025", "cirrus_bel/2025/**/*@DBZH.h5")):
        tot = sum(count(p) for p in sorted(glob.glob(str(DATA_DIR / pat), recursive=True)))
        n, a, b, c, fa, nf = tot
        print(f"{name}: {nf} files, {n:,} echo pixels; 55-70 dBZ {a:,} ({a / n:.2e}); >70 dBZ {b:,} ({b / n:.2e}), "
              f"of which >=100 dBZ {c:,}; files with any >70: {fa} ({fa / nf:.1%})")

## 4. Catalogue: frequency maps and `histo_cut`, per year

The catalogue of the review of #74, run here on the years on the PC (ODYSSEY 2023, CIRRUS 2025) and on every
ODYSSEY year once the 2012–2024 crops are there (`RUN_YEARS`). One pass per product and year stores, for each
pixel:

- **Frequency maps** (Wagner et al. 2012, section 3, Table 1): how often the pixel falls in each reflectivity
  class: 7–19, 19–28, 28–37, 37–46, 46–55 and ≥ 55 dBZ. The paper uses 1 dBZ instead of 7 in winter; 7 is
  kept all year here.
- **Annual sum** of the rain rate (Marshall–Palmer, capped at 55 dBZ; 15 min steps for ODYSSEY, 5 min for
  CIRRUS), and the number of valid times.
- **Times above 70 dBZ**, the discard threshold proposed in the review of #74.

Then `wradlib.classify.histo_cut` on the annual sum, with the 5% tails of EURADCLIM (Overeem et al. 2023,
section 3.3) instead of the 1% default: class 1 = persistently high (static clutter), class 2 = persistently
low (shading, blockage). Under the real-time rule the mask of one year is to be applied to the following
years, not to its own; here it is only mapped.

Results are cached in `RESULTS_DIR / "h2_artefacts" / "catalogue_<product>_<year>.npz"`; one ODYSSEY year
takes about 3 min, one CIRRUS year about 40 min.

In [ ]:
from wradlib.classify import histo_cut

CLASS_EDGES = [7, 19, 28, 37, 46, 55]     # Wagner et al. 2012, Table 1 (lower edges, dBZ)
HISTO_FREQ = 0.05                         # Overeem et al. 2023, section 3.3
CAT = RESULTS_DIR / "h2_artefacts"
CAT.mkdir(parents=True, exist_ok=True)
PRODUCTS = {"odyssey": ("odyssey_bel", "*.h5", 0.25), "cirrus": ("cirrus_bel", "*@DBZH.h5", 5 / 60)}
RUN_YEARS = {"odyssey": [2023], "cirrus": [2025]}     # after the 2012-2024 crops: "odyssey": list(range(2012, 2025))


def catalogue(product, year):
    path = CAT / f"catalogue_{product}_{year}.npz"
    if path.exists():
        return dict(np.load(path))
    folder, pattern, step_h = PRODUCTS[product]
    files = sorted(glob.glob(str(DATA_DIR / folder / str(year) / "**" / pattern), recursive=True))
    freq = acc = valid = gt70 = None
    for p in files:
        fr = read_odim_composite(p, "DBZH")
        ok = fr.coverage
        v = np.where(ok & ~fr.undetect, fr.values, -np.inf)
        if freq is None:
            freq = np.zeros((len(CLASS_EDGES),) + v.shape, np.int32)
            acc, valid, gt70 = np.zeros(v.shape), np.zeros(v.shape, np.int32), np.zeros(v.shape, np.int32)
        cls = np.digitize(v, CLASS_EDGES) - 1          # -1 below 7 dBZ
        for k in range(len(CLASS_EDGES)):
            freq[k] += cls == k
        acc += np.where(np.isfinite(v), rain(np.where(np.isfinite(v), v, 0.0)), 0.0) * step_h
        valid += ok
        gt70 += v > 70
    out = {"freq": freq, "annual_sum": acc, "valid": valid, "gt70": gt70, "n_files": np.array(len(files))}
    np.savez_compressed(path, **out)
    return out


cat = {}
for product, years in RUN_YEARS.items():
    for year in years:
        c = catalogue(product, year)
        mask = histo_cut(np.where(c["valid"] > 0, c["annual_sum"], np.nan), upper_frequency=HISTO_FREQ, lower_frequency=HISTO_FREQ)
        cat[(product, year)] = (c, mask)
        print(f"{product} {year}: {int(c['n_files'])} files; annual sum median {np.median(c['annual_sum'][c['valid'] > 0]):.0f} mm, "
              f"max {c['annual_sum'].max():.0f} mm; histo_cut static clutter {int((mask == 1).sum())} pixels, "
              f"shading {int((mask == 2).sum())} pixels; pixels above 70 dBZ at least once {int((c['gt70'] > 0).sum())}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

for (product, year), (c, mask) in cat.items():
    fig, ax = plt.subplots(1, 4, figsize=(22, 5.5), layout="constrained")
    im = ax[0].imshow(c["annual_sum"], cmap="viridis", vmax=np.percentile(c["annual_sum"], 99.5))
    ax[0].set_title(f"{product} {year}: annual sum (mm)"); fig.colorbar(im, ax=ax[0], shrink=0.8)
    im = ax[1].imshow(c["freq"][0] / np.maximum(c["valid"], 1), cmap="magma_r")
    ax[1].set_title("frequency of 7-19 dBZ"); fig.colorbar(im, ax=ax[1], shrink=0.8)
    im = ax[2].imshow(c["freq"][-1] / np.maximum(c["valid"], 1), cmap="magma_r", norm=LogNorm(vmin=1e-5, vmax=1))
    ax[2].set_title("frequency of >= 55 dBZ"); fig.colorbar(im, ax=ax[2], shrink=0.8)
    ax[3].imshow(np.where(mask > 0, mask, np.nan), cmap="coolwarm", vmin=1, vmax=2, interpolation="nearest")
    ax[3].set_title("histo_cut: static clutter (blue), shading (red)")
    for a_ in ax:
        a_.set_xticks([]); a_.set_yticks([])
    plt.show()

## What is seen

Run on 9/11/2026.

- **Summer convective days:** the filter removes 4.9% of the convective rain at 2 km (4.0–6.3% per day) and
  2.3% at 1 km (1.3–3.0%); 0.9% and 0.6% of the stratiform rain.
- **January days with stationary features:** it removes 11.2% of the convective rain at 2 km (9.2–16.4% per
  day) and 7.1% at 1 km (6.1–10.3%): more than twice the summer share at 2 km, three times at 1 km. These
  "convective" pixels are mostly the stationary features, but not only them, so this is an upper bound of
  the gain on the features, not a measure of it.
- **Where the removed convective rain lies:** 92–93% inside echo regions, 5–7% on edges, 1–3% in small
  echoes. The filter acts mostly through its gradient test, on pixels much stronger than their neighbours,
  not by removing small cells.
- **Catalogue, ODYSSEY 2023 and CIRRUS 2025** (section 4):
  - The annual sums (median 2,831 and 2,204 mm) are far above the rain that falls in the region, because
    a column-maximum reflectivity converted with Marshall–Palmer is not a rain estimate; they serve only as
    a statistic for `histo_cut`.
  - With 5% tails, `histo_cut` flags 8,148 ODYSSEY pixels (3.4% of the crop) and 31,851 CIRRUS pixels
    (3.3%) as static clutter, and no shading. Most flagged areas are discs centred on the radars, where the
    annual sum is highest, plus the patches on and off the Dutch coast; the discs may be the effect of
    distance from the radar on a maximum composite rather than clutter (not checked).
  - The frequency of ≥ 55 dBZ shows the lines over the North Sea in both products, and radial lines in the
    south-east of ODYSSEY.